<table style="width:100%; border-collapse:collapse; border-spacing:0; border:none; border-bottom:2px solid #e0e0e0; margin-bottom:16px; padding:0; margin-top:0;">
  <tr style="border:none; padding:0; margin:0;">
    <td style="width:144px; padding:16px 24px 16px 0; vertical-align:middle; border:none; outline:none; box-shadow:none;">
      <img src="https://github.com/TUIlmenauAMS/fs-ws2627-pinn-guitar/blob/main/assets/ams_group_tui.png?raw=1" width="120" style="border-radius:12px; display:block; margin:0;"/>
    </td>
    <td style="padding:16px 0 16px 0; vertical-align:middle; border:none; outline:none; box-shadow:none;">
      <h1 style="margin:0 0 4px 0; padding:0;">Forschungsseminar – Medieningenieurwissenschaften</h1>
      <h2 style="margin:0 0 8px 0; padding:0;">NB02: Analytical Modal Solution</h2>
      <p style="margin:0; padding:0;">
        <b>Institut für Medientechnik – Technische Universität Ilmenau</b><br>
        <a href="https://www.tu-ilmenau.de/universitaet/fakultaeten/fakultaet-elektrotechnik-und-informationstechnik/profil/institute-und-fachgebiete/fachgebiet-angewandte-mediensysteme" target="_blank">Institut für Medientechnik – AMS Group</a>
      </p>
      <p style="margin:4px 0 0 0; padding:0;">
        <b>Fachgebietsleiter:</b> Prof. Dr.-Ing. Gerald Schuller<br>
        <b>Wissenschaftlicher Mitarbeiter:</b> Renato Fera P. <a href="https://www.linkedin.com/in/renato-profeta/" target="_blank">LinkedIn</a>
      </p>
    </td>
  </tr>
</table>

In [ ]:
# ── Environment detection ──────────────────────────────────────────────────────
# Runs automatically. Installs packages on Colab; confirms venv on local/Codespaces.
import sys, subprocess

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _in_colab():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "numpy", "matplotlib", "scipy", "sympy"])
    print("✓ Colab — packages installed via pip.")
else:
    # Local VS Code or GitHub Codespaces:
    # packages are managed by uv (pyproject.toml). Run 'uv sync' once in the
    # repo root, then select the .venv interpreter as the kernel in VS Code.
    print("✓ Local / Codespaces — using uv-managed venv.")
# ──────────────────────────────────────────────────────────────────────────────

# NB02: Analytical Modal Solution

> **After this notebook you will be able to:**
> - Explain why the spatial eigenfunctions sin(nπx/L) are orthogonal, and why that matters
> - Derive the modal coefficient formula Bₙ from the initial condition, step by step
> - Compute modal coefficients both symbolically (closed form) and numerically (general method), and confirm they agree
> - Visualise the full modal series solution for an initial condition that is not a single pure mode
> - Explain and verify numerically how many modes are needed for a good approximation

In [ ]:
# ── SymPy setup — symbolic variables used throughout this notebook ────────────
import sympy as sp
from sympy import (symbols, Function, Eq, sin, cos, exp, pi,
                   diff, simplify, integrate, oo, Sum, Piecewise)
from IPython.display import display

# Independent variables (same roles as in NB01)
x, t = sp.symbols('x t', real=True)

# Physical parameters (same roles as in NB01)
L, alpha = sp.symbols('L alpha', positive=True)

# Mode indices — n is "the" mode we track; m is a second, independent dummy
# index used only for the orthogonality derivation below
n, m = sp.symbols('n m', positive=True, integer=True)

# Modal (Fourier) coefficient — same name as in NB01
B_n = sp.Symbol('B_n')

# A generic function standing for "the initial condition", before we pick one
f = sp.Function('f')

# The uniform initial temperature used in this notebook's example
T0 = sp.Symbol('T_0', positive=True)

# u is the unknown function of x and t, as in NB01
u = sp.Function('u')

# Enable LaTeX rendering in notebook output
sp.init_printing(use_latex='mathjax')

print("✓ SymPy symbolic variables defined. All subsequent cells can use them.")
# ──────────────────────────────────────────────────────────────────────────────

---

## 1. Recap: where NB01 left off

In NB01 we derived, by separation of variables, that every solution of the 1D heat
equation satisfying the Dirichlet boundary conditions $u(0,t) = u(L,t) = 0$ can be
written as a sum over independent "modes":

$$u(x, t) = \sum_{n=1}^{\infty} B_n \sin\!\left(\frac{n\pi x}{L}\right) e^{-\alpha (n\pi/L)^2 t}$$

Setting $t = 0$ shows that the coefficients $B_n$ must reproduce the initial condition
$f(x)$ as a sum of sine waves:

$$f(x) = \sum_{n=1}^{\infty} B_n \sin\!\left(\frac{n\pi x}{L}\right)$$

In NB01 we chose an initial condition that was *already* a single pure mode
($f(x) = \sin(\pi x/L)$), so $B_1 = 1$ and every other $B_n = 0$ — no real work was
needed. **That was a special case.** In general, the initial condition is some arbitrary
shape, and we need a systematic way to extract every $B_n$ from it. That is the question
this notebook answers.

In [ ]:
# ── The general series solution, as derived in NB01 ────────────────────────────
u_general = Sum(B_n * sin(n * pi * x / L) * exp(-alpha * (n * pi / L)**2 * t),
                (n, 1, oo))
print("General series solution (from NB01):")
display(Eq(u(x, t), u_general))

u_at_t0 = Sum(B_n * sin(n * pi * x / L), (n, 1, oo))
print("\nAt t = 0, this must equal the initial condition:")
display(Eq(f(x), u_at_t0))

print("\nQuestion: given an arbitrary f(x), how do we find every B_n?")
# ──────────────────────────────────────────────────────────────────────────────

---

## 2. Orthogonality of the modes

**The trick:** multiply both sides of

$$f(x) = \sum_{n=1}^{\infty} B_n \sin\!\left(\frac{n\pi x}{L}\right)$$

by $\sin(m\pi x/L)$ for some *fixed* mode number $m$, and integrate over the whole rod,
$x \in [0, L]$:

$$\int_0^L f(x)\sin\!\left(\frac{m\pi x}{L}\right) dx
  = \sum_{n=1}^{\infty} B_n \int_0^L \sin\!\left(\frac{n\pi x}{L}\right)\sin\!\left(\frac{m\pi x}{L}\right) dx$$

This looks more complicated, not less — until we evaluate the integral on the right.
It turns out that

$$\int_0^L \sin\!\left(\frac{n\pi x}{L}\right)\sin\!\left(\frac{m\pi x}{L}\right) dx
  = \begin{cases} 0, & n \neq m \\[4pt] L/2, & n = m \end{cases}$$

This property is called **orthogonality**: different modes are, in a precise
integral sense, "perpendicular" to each other — much like the x, y and z axes in 3D space
don't interfere with one another. Let's not just take this on faith — we verify it with
SymPy below for concrete numbers.

In [ ]:
# ── Verify orthogonality for concrete mode numbers ─────────────────────────────
L_sym = sp.Symbol('L', positive=True)

print("Case n ≠ m  (e.g. n=2, m=3):")
integrand_diff = sin(2 * pi * x / L_sym) * sin(3 * pi * x / L_sym)
result_diff = sp.integrate(integrand_diff, (x, 0, L_sym))
display(Eq(sp.Integral(integrand_diff, (x, 0, L_sym)), result_diff))
print("✓ Zero, as claimed." if sp.simplify(result_diff) == 0 else "✗ Unexpected result.")

print("\nCase n = m  (e.g. n=m=3):")
integrand_same = sin(3 * pi * x / L_sym)**2
result_same = sp.integrate(integrand_same, (x, 0, L_sym))
display(Eq(sp.Integral(integrand_same, (x, 0, L_sym)), result_same))
print(f"✓ Equals L/2, as claimed." if sp.simplify(result_same - L_sym/2) == 0
      else "✗ Unexpected result.")
# ──────────────────────────────────────────────────────────────────────────────

### 2.1 The coefficient formula

Because of orthogonality, **every term in the sum on the right vanishes except the one
where $n = m$.** The infinite sum collapses to a single surviving term:

$$\int_0^L f(x)\sin\!\left(\frac{m\pi x}{L}\right) dx = B_m \cdot \frac{L}{2}$$

Solving for $B_m$ (and renaming $m \to n$, since it was just a label):

$$\boxed{B_n = \frac{2}{L}\int_0^L f(x)\sin\!\left(\frac{n\pi x}{L}\right) dx}$$

This is the formula we were missing. It works for **any** initial condition $f(x)$ that
satisfies the boundary conditions reasonably well — we no longer need $f(x)$ to already
look like a sine wave.

In [ ]:
# ── The general modal coefficient formula ──────────────────────────────────────
Bn_formula = sp.Rational(2) / L * sp.Integral(f(x) * sin(n * pi * x / L), (x, 0, L))
print("General modal coefficient formula:")
display(Eq(B_n, Bn_formula))
# ──────────────────────────────────────────────────────────────────────────────

---

## 3. Visualising the mode shapes

NB01 worked with a single mode ($n=1$). Before computing coefficients for a general
initial condition, it helps to see the family of building blocks we are projecting onto:
the first several spatial eigenfunctions $\sin(n\pi x/L)$.

Notice that higher-$n$ modes oscillate more within the same rod length — and recall from
NB01 that their time-decay rate $\alpha(n\pi/L)^2$ grows with $n^2$. **Higher modes decay
much faster.** This will matter directly when we reconstruct the full solution later.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

L_num = 1.0
x_num = np.linspace(0, L_num, 300)

fig, ax = plt.subplots(figsize=(9, 4))
colors = plt.cm.viridis(np.linspace(0, 0.85, 5))
for n_val, col in zip(range(1, 6), colors):
    ax.plot(x_num, np.sin(n_val * np.pi * x_num / L_num), lw=2, color=col,
            label=f'n = {n_val}')
ax.axhline(0, color='k', lw=0.5)
ax.set(title=r'First 5 spatial modes  $\sin(n\pi x/L)$',
       xlabel='x  [m]', ylabel=r'$\sin(n\pi x/L)$')
ax.legend(title='Mode number')
plt.tight_layout()
plt.show()

---

## 4. A general initial condition: a uniformly hot rod, suddenly clamped to cold ends

**The scenario:** a rod has been sitting at a uniform temperature $T_0$ for a long time.
At $t = 0$, both ends are suddenly clamped to ice-cold reservoirs ($u = 0$). This is a
textbook-classic initial condition for the heat equation:

$$f(x) = T_0, \qquad 0 < x < L$$

Unlike NB01's initial condition, this is **not** a pure mode — it is a flat line, and no
single $\sin(n\pi x/L)$ shape looks like a flat line. We genuinely need the projection
formula from Section 2 here.

Notice something subtle: at the very endpoints $x=0$ and $x=L$, the initial condition
says $u = T_0 \neq 0$, while the boundary condition demands $u = 0$ for all $t>0$. This
mismatch at the two edges is exactly why infinitely many modes are required — and why, as
we will see later, a truncated reconstruction shows a little "ringing" right next to the
endpoints no matter how many modes we include.

In [ ]:
# ── The initial condition for this notebook ────────────────────────────────────
f_expr = T0  # constant on (0, L)
print("Initial condition:")
display(Eq(u(x, 0), f_expr))

print("\nDirichlet boundary conditions (same as NB01):")
display(Eq(u(0, t), 0))
display(Eq(u(L, t), 0))

print("\nNote: f(0) = f(L) = T_0, which does NOT equal the boundary value 0.")
print("This mismatch at the two endpoints is expected and will reappear later")
print("as mild ringing in the truncated reconstruction near x=0 and x=L.")
# ──────────────────────────────────────────────────────────────────────────────

---

## 5. Computing the coefficients — two ways

We now apply the formula from Section 2.1 to $f(x) = T_0$:

$$B_n = \frac{2}{L}\int_0^L T_0 \sin\!\left(\frac{n\pi x}{L}\right) dx$$

This particular integral happens to have a **closed form**. We derive it below the same
way a textbook would — antiderivative, then evaluate at the upper bound, then the lower
bound, then subtract — with SymPy actually computing each intermediate step, not just the
final answer. But most real initial conditions do not have a nice closed form, so we also
compute the same coefficients **numerically**, using `scipy.integrate.quad` — a method
that works for *any* initial condition, closed form or not. If both methods were
implemented correctly, they must agree.

In [ ]:
# ── Step 1: the indefinite integral (antiderivative), computed by SymPy ───────
F = sp.integrate(T0 * sin(n * pi * x / L), x)       # no bounds yet
print("Antiderivative of T₀ sin(nπx/L):")
display(Eq(sp.Integral(T0 * sin(n * pi * x / L), x), F))

# ── Step 2: evaluate at the upper bound x = L ──────────────────────────────────
F_upper = F.subs(x, L)
print("\nEvaluate at x = L:")
display(Eq(F.subs(x, sp.Symbol('L')), F_upper))

# ── Step 3: evaluate at the lower bound x = 0 ──────────────────────────────────
F_lower = F.subs(x, 0)
print("\nEvaluate at x = 0:")
display(Eq(F.subs(x, 0), F_lower))

# ── Step 4: subtract (Fundamental Theorem of Calculus) ─────────────────────────
definite_integral = sp.simplify(F_upper - F_lower)
print("\nSubtract (upper − lower):")
display(Eq(sp.Integral(T0 * sin(n * pi * x / L), (x, 0, L)), definite_integral))

# ── Step 5: multiply by the 2/L prefactor from the B_n formula ────────────────
Bn_result = sp.simplify(sp.Rational(2) / L * definite_integral)
print("\nMultiply by 2/L:")
display(Eq(B_n, Bn_result))

# ── Step 6: rewrite cos(nπ) as (-1)ⁿ to split even/odd n explicitly ───────────
Bn_simplified = Bn_result.rewrite(sp.cos).subs(cos(n * pi), (-1)**n)
Bn_simplified = sp.simplify(Bn_simplified)
print("\nUsing cos(nπ) = (-1)ⁿ :")
display(Eq(B_n, Bn_simplified))

print("\n→ For even n, (-1)ⁿ = 1, so B_n = 0.")
print("→ For odd n,  (-1)ⁿ = -1, so B_n = 4T₀/(nπ).")
print("Only the ODD harmonics survive.")
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── General numeric method: works for ANY initial condition ───────────────────
from scipy.integrate import quad

alpha_num = 0.01   # thermal diffusivity [m²/s]
T0_num    = 1.0    # uniform initial temperature
N_modes   = 15      # how many coefficients to compute for the comparison

def initial_condition(x):
    return T0_num * np.ones_like(x)

def modal_coeff_numeric(n_val):
    integrand = lambda xv: T0_num * np.sin(n_val * np.pi * xv / L_num)
    val, _ = quad(integrand, 0, L_num)
    return (2 / L_num) * val

def modal_coeff_closed_form(n_val):
    # B_n = 4 T0 / (n π) for odd n, 0 for even n
    return 0.0 if n_val % 2 == 0 else 4 * T0_num / (n_val * np.pi)

ns = np.arange(1, N_modes + 1)
Bn_numeric     = np.array([modal_coeff_numeric(n_val) for n_val in ns])
Bn_closed_form = np.array([modal_coeff_closed_form(n_val) for n_val in ns])

print(f"{'n':>3} {'closed form':>14} {'numeric (quad)':>16} {'difference':>14}")
for n_val, bf, bn in zip(ns, Bn_closed_form, Bn_numeric):
    print(f"{n_val:>3} {bf:>14.6f} {bn:>16.6f} {abs(bf - bn):>14.2e}")

In [ ]:
# ── Visual comparison: closed form vs. numeric ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.stem(ns, Bn_closed_form, linefmt='C0-', markerfmt='C0o', basefmt='k-',
        label='Closed form')
ax.scatter(ns + 0.15, Bn_numeric, facecolors='none', edgecolors='C3', s=70,
           linewidths=1.8, zorder=5, label='Numeric (quad)')
ax.set(title='Modal coefficients $B_n$ — closed form vs. numeric integration',
       xlabel='Mode number n', ylabel='$B_n$')
ax.axhline(0, color='k', lw=0.5)
ax.legend()
plt.tight_layout()
plt.show()

print("The two methods agree to numerical precision — confirming our symbolic",
      "derivation is correct, and that the general numeric method works even",
      "without knowing the closed form in advance.")

---

## 6. Assembling the modal solution

With all the coefficients in hand, we truncate the infinite series at some finite number
of modes $N$ and reconstruct an approximation of the full solution:

$$u(x,t) \approx \sum_{n=1}^{N} B_n \sin\!\left(\frac{n\pi x}{L}\right) e^{-\alpha (n\pi/L)^2 t}$$

Because $B_n = 0$ for every even $n$ in our example, only the odd terms contribute — but
we sum over all $n$ anyway, which automatically takes care of that.

In [ ]:
# ── The truncated modal solution, displayed symbolically ──────────────────────
N_sym = sp.Symbol('N', positive=True, integer=True)
u_truncated = Sum(B_n * sin(n * pi * x / L) * exp(-alpha * (n * pi / L)**2 * t),
                   (n, 1, N_sym))
print("Truncated modal solution (first N modes):")
display(Eq(u(x, t), u_truncated))

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
T_end   = 10.0   # end time for visualisation [s]
Nx, Nt  = 200, 300
N_terms = 50     # number of modes used in the reconstruction

x_num = np.linspace(0, L_num, Nx)
t_num = np.linspace(0, T_end, Nt)
X_grid, T_grid = np.meshgrid(x_num, t_num)

coeffs_full = np.array([modal_coeff_closed_form(nv) for nv in range(1, N_terms + 1)])

U = np.zeros_like(X_grid)
for n_val, Bn_val in zip(range(1, N_terms + 1), coeffs_full):
    decay = np.exp(-alpha_num * (n_val * np.pi / L_num)**2 * T_grid)
    U += Bn_val * np.sin(n_val * np.pi * X_grid / L_num) * decay

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

snapshot_times = [0, 0.5, 1, 2, 5, 10]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(snapshot_times)))
for t_snap, col in zip(snapshot_times, colors):
    idx = np.argmin(np.abs(t_num - t_snap))
    axes[0].plot(x_num, U[idx], color=col, lw=2, label=f't = {t_snap} s')
axes[0].set(title=f'Modal reconstruction ({N_terms} modes) — snapshots',
            xlabel='x  [m]', ylabel='u(x, t)')
axes[0].legend(fontsize=8)

im = axes[1].imshow(U, aspect='auto', origin='lower',
                    extent=[0, L_num, 0, T_end], cmap='hot')
plt.colorbar(im, ax=axes[1], label='u(x, t)')
axes[1].set(title='Modal reconstruction — space–time heatmap',
            xlabel='x  [m]', ylabel='t  [s]')

plt.tight_layout()
plt.show()

**What you should see:**
- **Snapshots (left):** at $t=0$ the curve is nearly flat at $T_0$, with a small amount
  of ringing right next to the two endpoints (a direct consequence of the mismatch
  between $f(0)=f(L)=T_0$ and the boundary value $0$, discussed in Section 4). As time
  increases, the whole profile relaxes smoothly toward $u=0$ everywhere, since heat
  continuously escapes through the cold ends.
- **Heatmap (right):** heat visibly drains out from both edges inward over time — the
  corners near $x=0$ and $x=L$ cool first, while the centre of the rod stays warm the
  longest.

> This "edges cool first, faster modes decay faster" behaviour is a direct, visible
> consequence of the $(n\pi/L)^2$ decay-rate dependence from NB01: the high-$n$ modes
> needed to represent the sharp corners near the endpoints die out almost immediately,
> while the slowly-decaying $n=1$ mode dominates the long-term shape.

---

## 7. Convergence: how many modes are enough?

Including more modes makes the truncated series a better approximation of the true
initial condition — but each additional mode contributes less, since the coefficients
$B_n = 4T_0/(n\pi)$ shrink as $n$ grows. We can measure "how good is the approximation"
with the **L² error**: the root-mean-square difference between the true, flat initial
condition and the N-mode reconstruction of it at $t=0$.

> Because our initial condition has a jump at the two endpoints (between $T_0$ and the
> boundary value $0$), the error near $x=0$ and $x=L$ shrinks more slowly than everywhere
> else, no matter how many modes we add — a mild hint of what is known as the Gibbs
> phenomenon. We will not develop this further here; it is simply why the overall error
> decreases steadily but not perfectly smoothly.

In [ ]:
# ── Convergence of the modal series reconstruction ─────────────────────────────
u0_true = np.full_like(x_num, T0_num)
mode_counts = [1, 2, 3, 5, 10, 15, 20, 30, 50, 100]
errors = []
for N in mode_counts:
    u0_approx = sum(modal_coeff_closed_form(nv) * np.sin(nv * np.pi * x_num / L_num)
                     for nv in range(1, N + 1))
    errors.append(np.sqrt(np.mean((u0_approx - u0_true)**2)))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(mode_counts, errors, 'o-', lw=2, color='C0')
ax.set(title='Convergence of the modal series (L² error vs. number of modes)',
       xlabel='Number of modes N', ylabel='L² error')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print("More modes → smaller error, but with diminishing returns:",
      "each extra mode contributes less than the one before it.")

---

## 8. Summary

**What you now know:**

1. The spatial eigenfunctions $\sin(n\pi x/L)$ are **orthogonal** — the integral of the
   product of two different modes over the rod is always zero.
2. Orthogonality lets us **project** any initial condition onto each mode individually,
   giving the general coefficient formula
   $B_n = \frac{2}{L}\int_0^L f(x)\sin(n\pi x/L)\,dx$ — derived here, not just stated.
3. For a uniformly hot rod suddenly clamped to cold ends, this integral has a clean
   closed form ($B_n = 4T_0/(n\pi)$ for odd $n$, zero for even $n$), and a general
   numerical method (`scipy.integrate.quad`) reproduces the same values for *any*
   initial condition, with or without a closed form.
4. The full modal solution is a **weighted sum of decaying standing waves**; because
   high-$n$ modes decay fastest, sharp features in the initial condition (like the
   corners near the clamped ends here) smooth out first, while the broad, low-$n$ shape
   persists longest.
5. More modes in the truncated sum give a better approximation, but with **diminishing
   returns** — later modes contribute less than earlier ones.

**Connection to the stiff guitar string project:**
The same orthogonality-and-projection logic applies directly to the stiff string
equation, once its own eigenfunctions and boundary conditions are known: students will
derive the string's eigenfunctions (modified by the stiffness term) and use exactly this
projection technique to find the modal coefficients for the string's own (plucked)
initial condition.

---

<div style="text-align:center; padding:18px 0 8px 0; margin-top:24px; border-top:2px solid #e0e0e0;">
  <p style="margin:0 0 6px 0;">
    <b>Forschungsseminar – Medieningenieurwissenschaften</b> &middot; NB02: Analytical Modal Solution<br>
    Institut für Medientechnik &ndash; Technische Universität Ilmenau
  </p>
  <p style="margin:0; font-size:0.95em;">
    Notebook created by <b>Renato Fera P.</b> &middot;
    <a href="https://www.linkedin.com/in/renato-profeta/" target="_blank">LinkedIn</a>
  </p>
</div>